# **ResNet-50 w/o Preprocessing**

> *Residual Neural Network (ResNet)*

In [ ]:
!pip install tensorflow --upgrade
!pip install keras --upgrade

## **Importing Libraries**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50

## **Loading the Dataset**

In [ ]:
dataset_path = '/content/drive/MyDrive/Resized Image'

subfolders = sorted(os.listdir(dataset_path))
print("Subfolders:", subfolders)

In [5]:
images = []
labels = []

for label, folder in enumerate(subfolders):
    folder_path = os.path.join(dataset_path, folder)

    for img_file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_file)
        img = cv2.imread(img_path)

        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(label)
        else:
            print(f"Failed to load image: {img_path}")

images = np.array(images)
labels = np.array(labels)

In [ ]:
print("Original shape of the images: ", images.shape)
print("Original shape of the labels: ", labels.shape)

In [7]:
images = images.astype('float32') / 255.0

X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)

In [ ]:
print("Shape of X train:", X_train.shape)
print("Shape of X test :", X_test.shape, '\n')

print("Shape of y train:", y_train.shape)
print("Shape of y test :", y_test.shape)

## **Model Training**

In [ ]:
input_shape = X_train.shape[1:]
num_classes = len(subfolders)

resnet50_base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
resnet50_base_model.trainable = False

model = models.Sequential([
    layers.Rescaling(scale=255.0),
    layers.Lambda(tf.keras.applications.resnet50.preprocess_input),
    resnet50_base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
resnet50_base_model.summary()

In [ ]:
model.summary()

In [ ]:
history = model.fit(X_train, y_train,
                    epochs=10, batch_size=32,
                    validation_split=0.2)

## **Model Testing**

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred_resnet50 = np.argmax(y_pred_probs, axis=1)

accuracy_resnet50 = accuracy_score(y_test, y_pred_resnet50)
print(f"ResNet50 Accuracy: {accuracy_resnet50:.4f}")

In [ ]:
average_precision = precision_score(y_test, y_pred_resnet50, average='macro')
average_recall = recall_score(y_test, y_pred_resnet50, average='macro')
average_f1 = f1_score(y_test, y_pred_resnet50, average='macro')

print(f"Average Precision: {average_precision:.4f}")
print(f"Average Recall: {average_recall:.4f}")
print(f"Average F1-Score: {average_f1:.4f}")

In [ ]:
print("ResNet50 Classification Report:\n")
print(classification_report(y_test, y_pred_resnet50, target_names=subfolders)) 

## **Visualisation**

### **Training and Validation Accuracy**

In [ ]:
plt.figure(figsize=(6, 4))

plt.title('Accuracy (ResNet50)')
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend()
plt.show()

### **Training and Validation Loss**

In [ ]:
plt.figure(figsize=(6, 4))

plt.title('Loss (ResNet50)')
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend()
plt.show()

### **Confusion Matrix**

In [ ]:
conf_matrix_resnet50 = confusion_matrix(y_test, y_pred_resnet50)

conf_matrix_resnet50_normalised = conf_matrix_resnet50.astype('float') / conf_matrix_resnet50.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(12, 10))
sns.heatmap(conf_matrix_resnet50_normalised, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=subfolders, yticklabels=subfolders)
plt.title('Normalized Confusion Matrix (ResNet50)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### **Class-wise Precision, Recall, and F1-Score**

In [ ]:
precision = precision_score(y_test, y_pred_resnet50, average=None)
recall = recall_score(y_test, y_pred_resnet50, average=None)
f1 = f1_score(y_test, y_pred_resnet50, average=None)

class_metrics = pd.DataFrame({
    'Class': subfolders,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1
})

print("Class-wise Precision, Recall, and F1-Score (ResNet50):\n")
print(class_metrics)

In [ ]:
plt.figure(figsize=(16, 8))
x = np.arange(len(subfolders))
width = 0.2

plt.bar(x - width, precision, width, label='Precision', color='lightcoral')
plt.bar(x, recall, width, label='Recall', color='palegreen')
plt.bar(x + width, f1, width, label='F1-Score', color='lightskyblue')

plt.title('Class-wise Precision, Recall, and F1-Score (ResNet50)')
plt.xlabel('Class')
plt.ylabel('Score')
plt.xticks(x, subfolders, rotation=90)
plt.legend()
plt.show()

## **Exporting Model**

In [ ]:
model.save('/content/drive/MyDrive/ResNet50_NoPreprocessing.h5')